# EDL+ORCU Fluorosis Diagnosis — Kaggle Training Notebook

**Tasks**: Dental Fluorosis (DF) + Skeletal Fluorosis (SF), 4-class ordinal classification.

**Kaggle GPU**: Select T4 x2 accelerator. Expected runtime ~6-8 hours for full experiments.

**Input datasets**: `fluorosis-data` (images + GT) and `fluorosis-code` (Python source).

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

# Clone code from GitHub (internet available during setup)
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

In [ ]:
# Kaggle paths
CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/fluorosis-data"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

## 2. Data Verification
Check that both DF and SF datasets are accessible with correct class distributions.

In [ ]:
def verify_data(task):
    """Verify dataset structure and print class distribution per split."""
    ds_cls = DFDataset if task == "df" else SFDataset
    for split in ["train", "val", "test"]:
        ds = ds_cls(DATA_ROOT, split=split)
        labels = np.array([ds[i][1] for i in range(len(ds))])
        dist = {f"class_{k}": int((labels == k).sum()) for k in range(4)}
        print(f"  {task.upper()} {split:5s}: {len(ds):3d} samples | {dist}")

print("=== Data Verification ===")
verify_data("df")
print()
verify_data("sf")
print("\nData OK.")

## 3. Training Functions
Self-contained training loop replicating `train.py`. Supports all 4 modes: `ce`, `edl`, `orcu`, `edl_orcu`.

In [ ]:
# ---- Mode-specific loss wrappers ----

class CEModeWrapper(nn.Module):
    """Standard cross-entropy on logits z."""
    def forward(self, alpha, z, targets, epoch=None):
        log_probs = F.log_softmax(z, dim=-1)
        loss = F.nll_loss(log_probs, targets)
        return loss, {"stage": 0, "L_ce": loss.item()}


class EDLModeWrapper(nn.Module):
    """EDL only (no ORCU)."""
    def __init__(self, num_classes, kl_lambda, total_epochs):
        super().__init__()
        from losses.edl_loss import EDLLoss
        self.edl = EDLLoss(num_classes=num_classes, kl_lambda=kl_lambda)
        self.total_epochs = total_epochs

    def forward(self, alpha, z, targets, epoch=None):
        loss = self.edl(alpha, targets, epoch, self.total_epochs)
        return loss, {"stage": 1, "L_edl": loss.item()}


class ORCUModeWrapper(nn.Module):
    """ORCU only (SORD-CE + log-barrier)."""
    def __init__(self, num_classes, t):
        super().__init__()
        from losses.orcu_loss import ORCULoss
        self.orcu = ORCULoss(num_classes=num_classes, t=t)

    def forward(self, alpha, z, targets, epoch=None):
        loss = self.orcu(z, targets)
        return loss, {"stage": 0, "L_orcu": loss.item()}

In [ ]:
# ---- Optimizer builder ----

def build_optimizer(model, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone"):
            backbone_params.append(p)
        else:
            head_params.append(p)
    return optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params, "lr": lr_head},
    ], weight_decay=weight_decay)


def build_scheduler(optimizer, epochs, warmup_epochs):
    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    return SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs])

In [ ]:
# ---- Core training function ----

@torch.no_grad()
def evaluate_model(model, dataloader):
    """Run full evaluation over a dataloader, returning compute_metrics dict."""
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)
        alpha, z = model(images)
        all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())

    alpha = torch.cat(all_alpha, dim=0)
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets)


def train_model(task, data_root, output_dir, mode="edl_orcu",
                batch_size=None, epochs=None, lr_backbone=1e-4, lr_head=1e-3,
                stage_1_epochs=None, stage_2_epochs=None,
                lambda_orcu=0.5, lambda_kl=0.1, orcu_t=3.0,
                warmup_epochs=None, weight_decay=0.05, seed=42):
    """
    Train one model configuration. Returns (test_metrics, history).
    Saves best.pt, last.pt, test_metrics.json, history.json to output_dir.
    """
    # Task defaults
    if task == "df":
        batch_size = batch_size or 32
        epochs = epochs or 100
        stage_1_epochs = stage_1_epochs or 5
        stage_2_epochs = stage_2_epochs or 30
        warmup_epochs = warmup_epochs or 5
    else:  # sf
        batch_size = batch_size or 16
        epochs = epochs or 150
        stage_1_epochs = stage_1_epochs or 10
        stage_2_epochs = stage_2_epochs or 45
        warmup_epochs = warmup_epochs or 10

    torch.manual_seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # Data
    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

    # Model
    model = build_model(task=task)
    model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params:,} params")

    # Criterion
    if mode == "ce":
        criterion = CEModeWrapper()
    elif mode == "edl":
        criterion = EDLModeWrapper(num_classes=4, kl_lambda=lambda_kl, total_epochs=epochs)
    elif mode == "orcu":
        criterion = ORCUModeWrapper(num_classes=4, t=orcu_t)
    else:  # edl_orcu
        criterion = CombinedLoss(
            num_classes=4, lambda_orcu=lambda_orcu, lambda_kl=lambda_kl, orcu_t=orcu_t,
            stage_1_epochs=stage_1_epochs, stage_2_epochs=stage_2_epochs, total_epochs=epochs)

    optimizer = build_optimizer(model, lr_backbone=lr_backbone, lr_head=lr_head, weight_decay=weight_decay)
    scheduler = build_scheduler(optimizer, epochs, warmup_epochs)

    # Training loop
    best_val_acc, best_state = 0.0, None
    history = []

    for epoch in range(epochs):
        model.train()
        epoch_losses = {"L_ce": 0.0, "L_edl": 0.0, "L_orcu": 0.0, "L_total": 0.0}
        epoch_stage, n_batches = 0, 0

        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            alpha, z = model(images)
            loss, loss_info = criterion(alpha, z, targets, epoch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_stage = loss_info.get("stage", 0)
            for k in epoch_losses:
                if k in loss_info:
                    epoch_losses[k] += loss_info[k]
            n_batches += 1

        scheduler.step()
        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        # Validation
        val_metrics = evaluate_model(model, val_loader)
        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "epoch": epoch, "stage": epoch_stage,
            "train_loss": epoch_losses,
            "val_metrics": {k: v for k, v in val_metrics.items()
                             if isinstance(v, (float, int)) and not k.startswith("evidence_class_")},
        })

        # Print every 10 epochs or at the end
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            loss_key = "L_total" if mode == "edl_orcu" and epoch_stage == 3 else \
                       ("L_ce" if mode == "ce" else ("L_edl" if mode == "edl" else "L_orcu"))
            print(f"[E{epoch+1:3d}/{epochs}] S={epoch_stage} | {loss_key}={epoch_losses.get(loss_key, 0):.4f} | "
                  f"Acc={val_metrics['acc']:.4f} F1={val_metrics['macro_f1']:.4f} "
                  f"QWK={val_metrics['qwk']:.4f} ECE={val_metrics['ece']:.4f} U={val_metrics['mean_uncertainty']:.3f}")

    # Load best and test
    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)

    # Save
    torch.save({"model_state_dict": best_state, "test_metrics": test_metrics},
               os.path.join(output_dir, "best.pt"))
    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump({k: v for k, v in test_metrics.items() if isinstance(v, (float, int))}, f, indent=2)
    with open(os.path.join(output_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n[Test] Acc={test_metrics['acc']:.4f} F1={test_metrics['macro_f1']:.4f} "
          f"QWK={test_metrics['qwk']:.4f} ECE={test_metrics['ece']:.4f} "
          f"Unimodal={test_metrics['pct_unimodal']:.2%} U-ECE={test_metrics['u_ece']:.4f} "
          f"AUROC(u)={test_metrics['auroc_u']:.4f}")
    return test_metrics, history

## 4. Skeletal Fluorosis (SF) Experiments
ResNet-50 backbone, 4 modes: CE baseline, EDL, ORCU, EDL+ORCU (full).

In [ ]:
sf_results = {}
sf_modes = ["ce", "edl", "orcu", "edl_orcu"]

for mode in sf_modes:
    print(f"\n{'='*60}")
    print(f"SF | Mode: {mode}")
    print(f"{'='*60}")
    out_dir = os.path.join(OUTPUT_DIR, f"sf_{mode}")
    metrics, _ = train_model(
        "sf", DATA_ROOT, out_dir, mode=mode,
        epochs=150, batch_size=16)
    sf_results[mode] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}

# Save SF results
with open(os.path.join(OUTPUT_DIR, "sf_all_results.json"), "w") as f:
    json.dump(sf_results, f, indent=2)
print("\n=== SF Results ===")
for mode, m in sf_results.items():
    print(f"  {mode:12s}: Acc={m['acc']:.4f} F1={m['macro_f1']:.4f} QWK={m['qwk']:.4f} ECE={m['ece']:.4f} AUROC(u)={m['auroc_u']:.4f}")

## 5. Dental Fluorosis (DF) Experiments
ViT-Base backbone, 4 modes: CE baseline, EDL, ORCU, EDL+ORCU (full).

In [ ]:
df_results = {}
df_modes = ["ce", "edl", "orcu", "edl_orcu"]

for mode in df_modes:
    print(f"\n{'='*60}")
    print(f"DF | Mode: {mode}")
    print(f"{'='*60}")
    out_dir = os.path.join(OUTPUT_DIR, f"df_{mode}")
    metrics, _ = train_model(
        "df", DATA_ROOT, out_dir, mode=mode,
        epochs=100, batch_size=32)
    df_results[mode] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}

# Save DF results
with open(os.path.join(OUTPUT_DIR, "df_all_results.json"), "w") as f:
    json.dump(df_results, f, indent=2)
print("\n=== DF Results ===")
for mode, m in df_results.items():
    print(f"  {mode:12s}: Acc={m['acc']:.4f} F1={m['macro_f1']:.4f} QWK={m['qwk']:.4f} ECE={m['ece']:.4f} AUROC(u)={m['auroc_u']:.4f}")

## 6. Lambda ORCU Sweep
Sweep λ ∈ {0.1, 0.3, 0.5, 0.7, 1.0} on the best-performing task backbone. Reduced epochs (50) for speed.

In [ ]:
lambda_values = [0.1, 0.3, 0.5, 0.7, 1.0]
lambda_results = {}

for lam in lambda_values:
    print(f"\n{'='*50}")
    print(f"Lambda Sweep | λ={lam}")
    print(f"{'='*50}")

    # Run for both tasks
    for task in ["df", "sf"]:
        out_dir = os.path.join(OUTPUT_DIR, f"lambda_{task}_{lam}")
        batch_size = 32 if task == "df" else 16
        metrics, _ = train_model(
            task, DATA_ROOT, out_dir, mode="edl_orcu",
            epochs=50, batch_size=batch_size, lambda_orcu=lam,
            stage_1_epochs=3, stage_2_epochs=15)
        key = f"{task}_lam_{lam}"
        lambda_results[key] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}
        print(f"  [{key}] Acc={metrics['acc']:.4f} QWK={metrics['qwk']:.4f} ECE={metrics['ece']:.4f}")

with open(os.path.join(OUTPUT_DIR, "lambda_sweep.json"), "w") as f:
    json.dump(lambda_results, f, indent=2)

print("\n=== Lambda Sweep Summary ===")
for task in ["df", "sf"]:
    print(f"\n{task.upper()}:")
    for lam in lambda_values:
        key = f"{task}_lam_{lam}"
        if key in lambda_results:
            m = lambda_results[key]
            print(f"  λ={lam:.1f}: Acc={m['acc']:.4f} QWK={m['qwk']:.4f} ECE={m['ece']:.4f}")

## 7. 5-Fold Cross-Validation
Stratified K-fold CV with paired t-tests between all method pairs.

In [ ]:
def run_cv(task, data_root, methods=None, n_folds=5, epochs=80, seed=42):
    """
    Run stratified K-fold CV for each method. Returns per-fold metrics + t-test results.

    Uses deterministic dataset splits (same seed -> same samples) so fold indices
    are consistent when we re-instantiate datasets with different transforms.
    """
    if methods is None:
        methods = ["ce", "edl", "edl_orcu"]

    # Create label-only dataset (no transforms needed) to get ground-truth labels
    if task == "df":
        label_ds = DFDataset(data_root, split="train", split_seed=seed)
    else:
        label_ds = SFDataset(data_root, split="train")
    labels = np.array([label_ds[i][1] for i in range(len(label_ds))])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_results = {m: [] for m in methods}
    batch_size = 32 if task == "df" else 16

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(label_ds)), labels)):
        print(f"\n{'='*50}")
        print(f"{task.upper()} CV Fold {fold_idx+1}/{n_folds}")
        print(f"{'='*50}")

        # Fresh dataset instances with fold-appropriate transforms (deterministic split)
        if task == "df":
            train_full = DFDataset(data_root, split="train", split_seed=seed)
            val_full   = DFDataset(data_root, split="train", split_seed=seed)
        else:
            train_full = SFDataset(data_root, split="train")
            val_full   = SFDataset(data_root, split="train")

        train_full.transform = get_transforms(task, is_train=True)
        val_full.transform   = get_transforms(task, is_train=False)

        train_subset = Subset(train_full, train_idx)
        val_subset   = Subset(val_full, val_idx)

        # Test set: pre-defined split, no folding
        if task == "df":
            test_ds = DFDataset(data_root, split="test")
        else:
            test_ds = SFDataset(data_root, split="test")
        test_ds.transform = get_transforms(task, is_train=False)

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds,      batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

        for method in methods:
            print(f"  [{method}] training...")
            torch.manual_seed(seed + fold_idx * 100)
            model = build_model(task=task)
            model.to(device)

            # Build criterion
            if method == "ce":
                criterion = CEModeWrapper()
            elif method == "edl":
                criterion = EDLModeWrapper(num_classes=4, kl_lambda=0.1, total_epochs=epochs)
            elif method == "orcu":
                criterion = ORCUModeWrapper(num_classes=4, t=3.0)
            else:
                criterion = CombinedLoss(
                    num_classes=4, lambda_orcu=0.5, lambda_kl=0.1, orcu_t=3.0,
                    stage_1_epochs=5, stage_2_epochs=25, total_epochs=epochs)

            optimizer = build_optimizer(model)
            scheduler = build_scheduler(optimizer, epochs, warmup_epochs=5)

            best_val_acc = 0.0
            best_state = None

            for epoch in range(epochs):
                model.train()
                for images, targets in train_loader:
                    images = images.to(device)
                    targets = targets.to(device)
                    alpha, z = model(images)
                    loss, _ = criterion(alpha, z, targets, epoch)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                scheduler.step()

                # Quick val check every 10 epochs, save best
                if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
                    model.eval()
                    val_metrics = evaluate_model(model, val_loader)
                    if val_metrics["acc"] > best_val_acc:
                        best_val_acc = val_metrics["acc"]
                        best_state = copy.deepcopy(model.state_dict())

            model.load_state_dict(best_state)
            test_metrics = evaluate_model(model, test_loader)
            fold_results[method].append({"fold": fold_idx, **{k: v for k, v in test_metrics.items() if isinstance(v, (float, int))}})
            print(f"    Acc={test_metrics['acc']:.4f} F1={test_metrics['macro_f1']:.4f} QWK={test_metrics['qwk']:.4f}")

    # Aggregate
    summary = {}
    metric_names = ["acc", "macro_f1", "qwk", "ece", "pct_unimodal", "u_ece", "auroc_u"]
    for method in methods:
        agg = {}
        for mn in metric_names:
            vals = [f[mn] for f in fold_results[method] if mn in f]
            agg[mn] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "values": vals}
        summary[method] = agg

    # Paired t-tests
    significance = {}
    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            if i >= j:
                continue
            pair_key = f"{m1}_vs_{m2}"
            significance[pair_key] = {}
            for mn in ["acc", "macro_f1", "qwk", "ece"]:
                v1 = [f[mn] for f in fold_results[m1]]
                v2 = [f[mn] for f in fold_results[m2]]
                t_stat, p_val = stats.ttest_rel(v1, v2)
                significance[pair_key][mn] = {"t_statistic": float(t_stat), "p_value": float(p_val), "significant": bool(p_val < 0.05)}

    return {"task": task, "k": n_folds, "methods": methods,
            "fold_results": {m: [{k: v for k, v in fr.items()} for fr in fold_results[m]] for m in methods},
            "summary": {m: {k: {"mean": v["mean"], "std": v["std"]} for k, v in sv.items()} for m, sv in summary.items()},
            "significance": significance}


# Run CV for both tasks
for task in ["df", "sf"]:
    print(f"\n{'#'*60}")
    print(f"# 5-Fold CV: {task.upper()}")
    print(f"{'#'*60}")
    cv_result = run_cv(task, DATA_ROOT, methods=["ce", "edl", "edl_orcu"], n_folds=5, epochs=80)
    with open(os.path.join(OUTPUT_DIR, f"cv_{task}.json"), "w") as f:
        json.dump(cv_result, f, indent=2)

    print(f"\n=== {task.upper()} CV Summary (mean +/- std) ===")
    for method, agg in cv_result["summary"].items():
        print(f"  {method:12s}: Acc={agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
              f"QWK={agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}  "
              f"ECE={agg['ece']['mean']:.4f}+/-{agg['ece']['std']:.4f}")

    print(f"\n  Significance (p < 0.05):")
    for pair, tests in cv_result["significance"].items():
        sigs = [f"{m}(p={v['p_value']:.3f})" for m, v in tests.items() if v['significant']]
        if sigs:
            print(f"    {pair}: {', '.join(sigs)}")

## 8. Results Summary
Compile all experiment results into a single summary file.

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

# Load all saved results
all_summary = {}

# Main experiments
for fname in ["sf_all_results.json", "df_all_results.json", "lambda_sweep.json"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        with open(fpath) as f:
            all_summary[fname.replace(".json", "")] = json.load(f)

# CV results
for task in ["df", "sf"]:
    fpath = os.path.join(OUTPUT_DIR, f"cv_{task}.json")
    if os.path.exists(fpath):
        with open(fpath) as f:
            cv = json.load(f)
            all_summary[f"cv_{task}"] = cv.get("summary", {})
            all_summary[f"cv_{task}_significance"] = cv.get("significance", {})

# Print main table
print("\n--- Main Experiments ---")
for exp_name in ["sf_all_results", "df_all_results"]:
    if exp_name in all_summary:
        print(f"\n{exp_name}:")
        results = all_summary[exp_name]
        header = f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}"
        print(header)
        print("-" * len(header))
        for mode, m in results.items():
            print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} {m.get('qwk',0):8.4f} "
                  f"{m.get('ece',0):8.4f} {m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} {m.get('pct_unimodal',0):7.2%}")

# Print CV summary
print("\n--- Cross-Validation ---")
for task in ["df", "sf"]:
    key = f"cv_{task}"
    if key in all_summary and all_summary[key]:
        print(f"\n{task.upper()} 5-Fold CV:")
        cv_summary = all_summary[key]
        # cv_summary might be the full result or just summary
        for method, agg in cv_summary.items():
            if isinstance(agg, dict) and "acc" in agg:
                acc_m, acc_s = agg['acc']['mean'], agg['acc']['std']
                qwk_m, qwk_s = agg['qwk']['mean'], agg['qwk']['std']
                print(f"  {method:12s}: Acc={acc_m:.4f}±{acc_s:.4f}  QWK={qwk_m:.4f}±{qwk_s:.4f}")

# Save master summary
with open(os.path.join(OUTPUT_DIR, "master_summary.json"), "w") as f:
    json.dump(all_summary, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}/master_summary.json")
print("\n=== DONE ===")
print("Download the /kaggle/working/ directory for all model checkpoints and result files.")

---
## Appendix: Kaggle Submission Instructions

1. **Upload data**: Run `prepare_kaggle.sh` locally, then upload `kaggle_fluorosis_data.zip` as a **private** Kaggle Dataset named `fluorosis-data`.
2. **Push code**: Code is stored on GitHub (`XiaoHG/fluorosis_project`). The notebook clones it automatically during setup. If the repo is private, replace the HTTPS URL with a token-authenticated one: `https://<token>@github.com/XiaoHG/fluorosis_project.git`.
3. **Create notebook**: Upload this `.ipynb` file to Kaggle.
4. **Add data source**: In the notebook editor, click "Add Data" → search for `fluorosis-data`.
5. **Select accelerator**: Settings → Accelerator → T4 x2 (GPU).
6. **Run all**: Click "Run All" or execute cells in order.
7. **Download results**: After completion, download everything from the Output (/kaggle/working) panel.

**Expected runtime**: ~6-8 hours for all experiments (SF 4x150ep, DF 4x100ep, Lambda 10x50ep, CV 12x80ep).
Adjust epoch counts if runtime exceeds Kaggle's 9-hour session limit.